# Análisis de siniestros viales en Bogotá D.C. (2015–2021)
Dataset: `historico_siniestros_bogota_d_c_-.csv` (~199 mil registros).

**Preguntas que responde este cuaderno**
1. ¿Cómo ha evolucionado la siniestralidad por año y mes?
2. ¿Qué tan graves son los siniestros y de qué tipo?
3. ¿Qué localidades y vías concentran más casos y más muertes?
4. ¿En qué días y horas ocurren?
5. ¿Dónde están los puntos críticos (mapa de calor)?

## 1. Configuración y carga de datos

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
pd.options.display.float_format = "{:,.2f}".format

Sube el archivo CSV (también puedes montarlo desde Google Drive).

In [ ]:
from google.colab import files
subido = files.upload()          # selecciona historico_siniestros_bogota_d_c_-.csv
ruta = list(subido.keys())[0]

# Alternativa con Drive:
# from google.colab import drive; drive.mount('/content/drive')
# ruta = '/content/drive/MyDrive/historico_siniestros_bogota_d_c_-.csv'

df = pd.read_csv(ruta, encoding="utf-8-sig")   # utf-8-sig quita el BOM de la columna X
print(df.shape)
df.head()

## 2. Exploración y calidad de datos

In [ ]:
df.info()
print("\nNulos por columna:")
print(df.isna().sum()[lambda s: s > 0])
print("\nDuplicados en CODIGO_ACCIDENTE:", df.duplicated("CODIGO_ACCIDENTE").sum())

Hallazgos esperados: `PK_CALZADA` tiene ~19 % de nulos, `CIV` ~1 % y `LOCALIDAD` unos pocos. Las columnas `X/Y` y `LONGITUD/LATITUD` son redundantes.

## 3. Limpieza y variables derivadas

In [ ]:
df = df.drop(columns=["X", "Y"])   # duplican LONGITUD / LATITUD

# Fecha y hora (el sufijo +00 es solo formato; la hora es la hora local registrada)
df["fecha_hora"] = pd.to_datetime(df["FECHA_HORA_ACC"].str[:19], format="%Y/%m/%d %H:%M:%S")
df["anio"] = df["fecha_hora"].dt.year
df["mes"] = df["fecha_hora"].dt.month
df["hora"] = df["fecha_hora"].dt.hour
df["dia_semana"] = df["fecha_hora"].dt.dayofweek          # 0 = lunes
dias = ["Lunes","Martes","Miércoles","Jueves","Viernes","Sábado","Domingo"]
df["dia_nombre"] = df["dia_semana"].map(dict(enumerate(dias)))
df["fin_de_semana"] = df["dia_semana"] >= 5

df["LOCALIDAD"] = df["LOCALIDAD"].fillna("SIN DATO")
df["gravedad"] = df["GRAVEDAD"].str.title()
df["clase"] = df["CLASE_ACC"].str.title()

# Indicador de gravedad
df["con_muertos"] = df["GRAVEDAD"].eq("CON MUERTOS")
df["con_heridos"] = df["GRAVEDAD"].eq("CON HERIDOS")

# Cobertura temporal: 2021 llega hasta septiembre
print("Rango:", df["fecha_hora"].min(), "→", df["fecha_hora"].max())
df.head()

## 4. Evolución temporal

In [ ]:
# 2021 está incompleto (hasta el 10 de septiembre) y 2020 fue atípico por la pandemia
por_anio = df.groupby("anio").size()
ax = por_anio.plot(kind="bar", color="#3b6ea8")
ax.set_title("Siniestros por año (2021 incompleto)")
ax.set_xlabel(""); ax.set_ylabel("Siniestros")
for i, v in enumerate(por_anio):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
plt.show()

In [ ]:
# Serie mensual
mensual = df.set_index("fecha_hora").resample("MS").size()
ax = mensual.plot(color="#3b6ea8", lw=1.8)
ax.set_title("Siniestros por mes")
ax.set_ylabel("Siniestros"); ax.set_xlabel("")
ax.axvspan(pd.Timestamp("2020-03-25"), pd.Timestamp("2020-09-01"), color="red", alpha=0.1,
           label="Cuarentena estricta / reapertura (aprox.)")
ax.legend(); plt.show()

In [ ]:
# Estacionalidad: promedio mensual solo con años completos (2015-2020)
completos = df[df["anio"] <= 2020]
tabla = completos.pivot_table(index="mes", columns="anio", values="CODIGO_ACCIDENTE", aggfunc="count")
plt.figure(figsize=(10, 5))
sns.heatmap(tabla, annot=True, fmt=",.0f", cmap="YlOrRd")
plt.title("Siniestros por mes y año"); plt.show()

## 5. Gravedad y tipo de siniestro

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

g = df["gravedad"].value_counts()
axes[0].pie(g, labels=g.index, autopct="%1.1f%%", startangle=90,
            colors=["#9bb7d4", "#f2a65a", "#c0392b"])
axes[0].set_title("Gravedad")

c = df["clase"].value_counts()
sns.barplot(x=c.values, y=c.index, ax=axes[1], color="#3b6ea8")
axes[1].set_title("Clase de siniestro"); axes[1].set_xscale("log")
axes[1].set_xlabel("Siniestros (escala log)")
plt.tight_layout(); plt.show()

In [ ]:
# ¿Qué tipo de siniestro es más letal? (% con muertos y % con heridos)
letal = df.groupby("clase").agg(total=("clase","size"),
                                pct_muertos=("con_muertos","mean"),
                                pct_heridos=("con_heridos","mean"))
letal[["pct_muertos","pct_heridos"]] *= 100
letal.sort_values("pct_muertos", ascending=False)

In [ ]:
# Evolución de siniestros con muertos y heridos por año
ev = df.groupby("anio")[["con_muertos","con_heridos"]].sum()
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ev["con_muertos"].plot(marker="o", ax=ax[0], color="#c0392b"); ax[0].set_title("Siniestros con muertos")
ev["con_heridos"].plot(marker="o", ax=ax[1], color="#f2a65a"); ax[1].set_title("Siniestros con heridos")
plt.tight_layout(); plt.show()

## 6. Análisis por localidad

In [ ]:
loc = df[df["LOCALIDAD"] != "SIN DATO"].groupby("LOCALIDAD").agg(
    total=("clase","size"), muertos=("con_muertos","sum"), heridos=("con_heridos","sum"))
loc["tasa_letalidad_x1000"] = loc["muertos"] / loc["total"] * 1000
loc = loc[loc["total"] >= 100]   # excluye Sumapaz (solo 7 casos, tasa no representativa)
loc = loc.sort_values("total", ascending=False)
display(loc)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
loc["total"].sort_values().plot(kind="barh", ax=ax[0], color="#3b6ea8")
ax[0].set_title("Total de siniestros por localidad")
loc["tasa_letalidad_x1000"].sort_values().plot(kind="barh", ax=ax[1], color="#c0392b")
ax[1].set_title("Siniestros con muertos por cada 1.000")
plt.tight_layout(); plt.show()

In [ ]:
# Tipo de siniestro por localidad (% de cada fila)
cruce = pd.crosstab(df["LOCALIDAD"], df["clase"], normalize="index") * 100
plt.figure(figsize=(10, 8))
sns.heatmap(cruce[["Choque","Atropello","Caida De Ocupante","Volcamiento"]],
            annot=True, fmt=".1f", cmap="Blues")
plt.title("Distribución del tipo de siniestro por localidad (%)"); plt.show()

## 7. Cuándo ocurren: día y hora

In [ ]:
orden = dias
por_dia = df["dia_nombre"].value_counts().reindex(orden)
ax = por_dia.plot(kind="bar", color="#3b6ea8"); ax.set_title("Siniestros por día de la semana")
ax.set_xlabel(""); plt.xticks(rotation=30); plt.show()

In [ ]:
h = df.pivot_table(index="dia_nombre", columns="hora", values="CODIGO_ACCIDENTE", aggfunc="count").reindex(orden)
plt.figure(figsize=(14, 4.5))
sns.heatmap(h, cmap="YlOrRd", cbar_kws={"label": "Siniestros"})
plt.title("Siniestros por día y hora"); plt.xlabel("Hora del día"); plt.ylabel("")
plt.show()

In [ ]:
# ¿A qué horas son más graves? % con muertos por hora (entre semana vs fin de semana)
graves = df.groupby(["hora","fin_de_semana"])["con_muertos"].mean().unstack() * 100
graves.columns = ["Entre semana", "Fin de semana"]
graves.plot(figsize=(11, 4.5), marker="o"); plt.title("% de siniestros con muertos según la hora")
plt.ylabel("%"); plt.xticks(range(0, 24)); plt.show()

## 8. Vías con más siniestros

In [ ]:
# La dirección viene como 'AV AVENIDA BOYACA-CL 79 02' (intersección o tramo)
# Aproximación: usamos el primer eje vial antes del guion
df["via_principal"] = (df["DIRECCION"].str.split("-").str[0]
                       .str.replace(r"\s+", " ", regex=True).str.strip())
top_vias = df["via_principal"].value_counts().head(15)
top_vias.sort_values().plot(kind="barh", figsize=(10, 6), color="#3b6ea8")
plt.title("15 vías con más siniestros (aprox.)"); plt.show()

In [ ]:
# Puntos críticos: tramos viales (CIV) con más siniestros y más muertos
civ = (df[df["CIV"].fillna(0) > 0].groupby("CIV")   # CIV = 0 es un valor placeholder
         .agg(siniestros=("clase","size"), muertos=("con_muertos","sum"),
              direccion=("DIRECCION","first"), localidad=("LOCALIDAD","first"))
         .sort_values("siniestros", ascending=False))
civ.index = civ.index.astype(int)
display(civ.head(15))

## 9. Mapa de calor de siniestros

In [ ]:
# Dispersión simple (sin dependencias)
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].hexbin(df["LONGITUD"], df["LATITUD"], gridsize=80, cmap="inferno", mincnt=1, bins="log")
ax[0].set_title("Densidad de todos los siniestros")
m = df[df["con_muertos"]]
ax[1].scatter(df["LONGITUD"], df["LATITUD"], s=0.3, c="lightgray", alpha=0.3)
ax[1].scatter(m["LONGITUD"], m["LATITUD"], s=6, c="#c0392b", alpha=0.6)
ax[1].set_title("Siniestros con muertos")
for a in ax: a.set_xlabel("Longitud"); a.set_ylabel("Latitud"); a.set_aspect("equal")
plt.tight_layout(); plt.show()

In [ ]:
# Mapa interactivo (folium viene instalado en Colab)
import folium
from folium.plugins import HeatMap

mapa = folium.Map(location=[4.65, -74.1], zoom_start=11, tiles="CartoDB positron")
HeatMap(df[["LATITUD","LONGITUD"]].sample(60000, random_state=1).values.tolist(),
        radius=8, blur=10).add_to(mapa)
mapa

In [ ]:
# Mapa de siniestros con muertos, año 2019 (ejemplo; cambia el filtro)
mapa2 = folium.Map(location=[4.65, -74.1], zoom_start=11, tiles="CartoDB positron")
sub = df[(df["con_muertos"]) & (df["anio"] == 2019)]
for _, r in sub.iterrows():
    folium.CircleMarker([r["LATITUD"], r["LONGITUD"]], radius=3, color="#c0392b",
                        fill=True, popup=f'{r["DIRECCION"]} | {r["clase"]}').add_to(mapa2)
mapa2

## 10. Atropellos (usuarios vulnerables)

In [ ]:
atr = df[df["clase"] == "Atropello"]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
atr.groupby("anio").size().plot(kind="bar", ax=ax[0], color="#3b6ea8"); ax[0].set_title("Atropellos por año")
atr["LOCALIDAD"].value_counts().head(8).sort_values().plot(kind="barh", ax=ax[1], color="#3b6ea8")
ax[1].set_title("Localidades con más atropellos")
atr.groupby("hora").size().plot(ax=ax[2], marker="o"); ax[2].set_title("Atropellos por hora")
plt.tight_layout(); plt.show()
print(f"Atropellos: {len(atr):,} ({len(atr)/len(df):.1%} del total) | "
      f"con muertos: {atr['con_muertos'].mean():.1%} vs {df['con_muertos'].mean():.1%} general")

## 11. Modelo opcional: ¿qué predice que un siniestro sea fatal?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = pd.get_dummies(df[["clase","LOCALIDAD"]]).join(df[["hora","dia_semana","mes","anio"]])
y = df["con_muertos"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=200, max_depth=12, class_weight="balanced",
                            n_jobs=-1, random_state=42).fit(Xtr, ytr)
print(classification_report(yte, rf.predict(Xte), digits=3))

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)
imp.sort_values().plot(kind="barh", figsize=(8, 5), color="#3b6ea8")
plt.title("Variables más importantes"); plt.show()

## 12. Conclusiones
Completa con tus propios hallazgos tras ejecutar el cuaderno, por ejemplo:
- Tendencia anual y efecto de la pandemia en 2020.
- Localidades con mayor volumen vs. mayor letalidad.
- Franjas horarias críticas.
- Tipos de siniestro más letales (atropellos, volcamientos).
- Corredores viales y tramos (CIV) prioritarios.

**Limitaciones:** el dataset solo registra siniestros reportados, no tiene número de víctimas por evento, ni tipo de vehículo, y 2021 está incompleto.